# Mark 1 → Mark 4E — Ordered Research-Pipeline Notebook Suite

[![Phase](https://img.shields.io/badge/Research%20Phase-Mark%201%20to%204E-blue.svg)]()
[![Status](https://img.shields.io/badge/Mark%204E-Controlling%20Pass-success.svg)]()

## What this suite is

The eight research experiments previously run as separate notebooks in `mark 1/`
(`mark_1_probability_contrast_localization_diagnostic.ipynb` → `mark_4e_checkpoint_fusion_validation.ipynb`)
are reproduced here as **ten ordered, self-contained notebooks**. Each notebook answers one research
question, writes into the **same organized output root**, and ships its own visualizations — so the
whole loop can be re-run step by step (or a single phase re-run in isolation).

**Run order:** `00 → 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09`.

| # | Notebook | Phase |
|---|---|---|
| 00 | `00_pipeline_overview_and_setup.ipynb` | Overview + shared preflight (this notebook) |
| 01 | `01_mark_1_probability_contrast_localization_diagnostic.ipynb` | Mark 1 — Probability, Contrast & Localization Diagnostic |
| 02 | `02_mark_2_roi_multiwindow_feasibility.ipynb` | Mark 2 — Predicted-Liver ROI & Multi-Window Feasibility |
| 03 | `03_mark_3_two_stage_multiwindow_overfit.ipynb` | Mark 3 — Gated Two-Stage Multi-Window Overfit |
| 04 | `04_mark_4_two_stage_validation_smoke.ipynb` | Mark 4 — Two-Stage Multi-Window Validation Smoke |
| 05 | `05_mark_4b_roi_probability_diagnostics.ipynb` | Mark 4B — ROI Probability & Threshold Diagnostics |
| 06 | `06_mark_4c_two_channel_recall_ablation.ipynb` | Mark 4C — Two-Channel vs Recall-Loss Ablation |
| 07 | `07_mark_4d_metric_reconciliation_v116_diagnostic.ipynb` | Mark 4D — Metric Reconciliation & V116 Diagnostic |
| 08 | `08_mark_4e_checkpoint_fusion_validation.ipynb` | Mark 4E — Checkpoint Fusion Validation |
| 09 | `09_consolidated_results_and_reproduction_verification.ipynb` | Consolidated Results & Reproduction Verification |

## Pipeline flow

```mermaid
flowchart LR
  subgraph SETUP[Notebook 00 — shared preflight]
    P[provenance hashes + test lock]
  end
  P --> M1["01 Mark 1 — diagnostic (calibration FAIL)"]
  M1 -->|gate json| M2["02 Mark 2 — ROI feasibility (PASS)"]
  M2 -->|gate json| M3["03 Mark 3 — overfit gate (PASS)"]
  M3 -->|gate json| M4["04 Mark 4 — smoke (5/6 targets)"]
  M4 -->|gate json| M4B["05 Mark 4B — threshold calibration"]
  M4B -->|gate json| M4C["06 Mark 4C — two-channel vs recall-loss"]
  M4C -->|gate json| M4D["07 Mark 4D — reconciliation (V116 failure)"]
  M4D -->|gate json| M4E["08 Mark 4E — fusion (all 6 PASS)"]
  M4E -->|all 8 gates| C["09 Consolidation + reproduction verification"]
  style C fill:#d9f2e6,stroke:#1a7f4b

## Output folder map (shared by every notebook)

```
Evaluation/mark_1_to_4e_outputs/
├── mark_1_outputs/   … mark_1_gate_result.json, calibration_*, probability_*, hu_contrast_*
├── mark_2_outputs/   … mark_2_gate_result.json, roi_*, multiwindow_*, selected_roi_*
├── mark_3_outputs/   … mark_3_gate_result.json, overfit_*, roundtrip_*, training_roi_*
├── mark_4_outputs/   … mark_4_gate_result.json, mark_4_history.csv, best_*, mark_4_smoke_*
├── mark_4b_outputs/  … mark_4b_gate_result.json, threshold_*, calibration_*, focus_*
├── mark_4c_outputs/  … mark_4c_gate_result.json, arm_*, ablation_*, two_channel_*
├── mark_4d_outputs/  … mark_4d_gate_result.json, reconciled_*, v116_*, positive_patient_*
├── mark_4e_outputs/  … mark_4e_gate_result.json, fusion_*, selected_fusion_*
└── consolidated/     … unified_gate_summary.csv, reproduction_verification.json, status strip
```

Naming is uniform across phases: `<phase>_gate_result.json` for gates, `<phase>_history.csv` for
training histories, `*_metrics.csv` for patient/slice tables, and descriptive `<theme>_<name>.png`
for visualizations.

## Key results per phase

| # | Phase | Key result |
|---|---|---|
| 01 | Mark 1 diagnostic | calibration gate failed — signal mislocalized/suppressed |
| 02 | Mark 2 ROI feasibility | ROI contains ~100% tumour, median crop ≈ 42.7% |
| 03 | Mark 3 overfit gate | broad-1ch overfit hard Dice ≈ 0.9006 |
| 04 | Mark 4 validation smoke | 5/6 continuation targets met |
| 05 | Mark 4B calibration | threshold alone cannot fix recall |
| 06 | Mark 4C ablation | neither arm passed the full gate alone |
| 07 | Mark 4D reconciliation | V116 is a localization failure, not ROI clipping |
| 08 | Mark 4E fusion | maximum fusion @ 0.70 passes all 6 targets |
| 09 | Consolidated | 8/8 gates reproduced, unified summary written |

## Execution modes

- **REUSE mode (default, fast):** each notebook loads the frozen probability caches, ROI manifests,
  history CSVs and checkpoints from `mark 1/`, then recomputes every metric, gate and figure from those
  artifacts. Deterministic — the gate JSONs reproduce the originals exactly (drift < 1e-4, asserted).
- **REBUILD mode:** set the `REUSE_*` / `RUN_*` flags in the shared setup cell to `True` to re-run
  validation inference (rebuild `.npz` caches) and/or retrain the models from scratch (GPU hours).

**The test split stays locked in both modes** (opening it raises `PermissionError`).

> **How to run:** open `00_pipeline_overview_and_setup.ipynb` and select the Python (.venv) kernel,
> then `Run All`. Proceed through **01 → 09 in order** — phases 02–09 read the prior phase's gate JSON
> from the shared output folder, so the order is a real dependency. Each notebook embeds the shared
> setup cell; a phase can be re-run standalone once its upstream gates exist.

## Global setup

Everything below mirrors the shared preamble of the eight original notebooks: paths, seeds, hashes,
targets, and the framework modules (`VerifiedManifestDataset`, `MobileNetV2UNet`, `FocalDiceLoss`) are
verified before any computation starts. This cell is embedded verbatim in every notebook of the suite.

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "00_setup"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


In [2]:
# ---- Centralized output skeleton (new layout under Evaluation/output) ----
print("Centralized output skeleton under Evaluation/output/:")
for phase, folder in sorted(PHASE_DIR.items()):
    for sub in ("data", "figures", "caches"):
        (SHARED_OUTPUT_ROOT / folder / sub).mkdir(parents=True, exist_ok=True)
        print("  " + str(SHARED_OUTPUT_ROOT / folder / sub))
print(f"Shared artifact index: {ARTIFACT_INDEX}")


Centralized output skeleton under Evaluation/output/:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\00_pipeline_overview\data
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\00_pipeline_overview\figures
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\00_pipeline_overview\caches
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\09_consolidated\data
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\09_consolidated\figures
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\09_consolidated\caches
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\01_mark_1\data
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\01_mark_1\figures
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\01_mark_1\caches
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\02_mark_2\data
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\02_mark_2\figur

## How to read the phase notebooks

Every phase notebook (01–09) follows the same pattern:

1. **Intro** — the research question, its position in the pipeline, inputs, exact output file names,
   visualizations, expected gate, and the reproduced key finding.
2. **Shared setup** — identical global-setup cell (provenance + test lock + helpers).
3. **Phase cells** — verify provenance, reuse or rebuild artifacts, recompute metrics/gates, save CSVs,
   JSONs and figures, write `<phase>_gate_result.json` into the shared output root.
4. **Reproduction check** — compare the recomputed gate against the original `mark 1/mark_*_outputs/`
   gate and assert drift < 1e-4.

Notebook 09 consolidates every gate into one table and writes the verification report.